# Principal Component Analysis — Exercises

8 exercises covering the full PCA curriculum: from first-principles derivation through kernel PCA and LoRA rank analysis.

| Format | Description |
| --- | --- |
| **Problem** | Markdown cell with task description |
| **Your Solution** | Code scaffold — replace `# YOUR CODE HERE` |
| **Solution** | Reference solution with PASS/FAIL checks |

### Difficulty Levels

| Level | Exercises | Focus |
| --- | --- | --- |
| ★ | 1–3 | Core mechanics |
| ★★ | 4–6 | Deeper theory |
| ★★★ | 7–8 | AI / ML applications |

### Topic Map

| Topic | Exercise |
| --- | --- |
| PCA from eigendecomposition vs SVD | 1, 2 |
| Variance analysis and component selection | 3 |
| Whitening (PCA and ZCA) | 4 |
| Probabilistic PCA | 5 |
| Kernel PCA | 6 |
| LoRA and low-rank hypothesis | 7 |
| LLM intrinsic dimensionality | 8 |


In [ ]:
import numpy as np
import numpy.linalg as la

try:
    import matplotlib.pyplot as plt
    plt.style.use('seaborn-v0_8-whitegrid')
    plt.rcParams['figure.figsize'] = [10, 5]
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

np.set_printoptions(precision=6, suppress=True)
np.random.seed(42)

def header(title):
    print('\n' + '=' * len(title))
    print(title)
    print('=' * len(title))

def check_close(name, got, expected, tol=1e-6):
    ok = np.allclose(got, expected, atol=tol, rtol=tol)
    print(f"{'PASS' if ok else 'FAIL'} — {name}")
    if not ok:
        print(f'  expected: {np.round(expected, 6)}')
        print(f'  got     : {np.round(got, 6)}')
    return ok

def check_true(name, cond):
    print(f"{'PASS' if cond else 'FAIL'} — {name}")
    return cond

def pca_svd(X, k):
    """Reference PCA implementation via SVD."""
    mean = X.mean(axis=0)
    X_c = X - mean
    U, s, Vt = la.svd(X_c, full_matrices=False)
    return {
        'components': Vt[:k].T,
        'scores': U[:, :k] * s[:k],
        'explained_var': s[:k]**2 / (len(X) - 1),
        'explained_var_ratio': s[:k]**2 / (s**2).sum(),
        'singular_values': s,
        'mean': mean,
    }

print('Setup complete.')


---

## Exercise 1 ★ — PCA from Eigendecomposition vs SVD

**Goal:** Implement PCA two ways and verify they agree.

Given data matrix `X` (100 × 20):

**(a)** Center the data and compute the sample covariance matrix $C$.

**(b)** Compute the eigendecomposition of $C$ using `np.linalg.eigh` (symmetric solver). Sort eigenvalues in descending order.

**(c)** Compute the thin SVD of the centered data matrix.

**(d)** Show that the eigenvalues from (b) match $\sigma_i^2/(n-1)$ from (c).

**(e)** Show that the top-5 eigenvectors from (b) match the top-5 right singular vectors from (c) (up to sign: compare `|v1 · v2|`).

In [ ]:
# Exercise 1: Your Solution

np.random.seed(42)
n, d = 100, 20
# Structured data: decreasing variance along each dimension
X = np.random.randn(n, d) * np.arange(d, 0, -1)**0.5

def pca_eig(X, k):
    """PCA via covariance eigendecomposition."""
    # YOUR CODE HERE
    pass

def pca_via_svd(X, k):
    """PCA via SVD on centered data."""
    # YOUR CODE HERE
    pass

eig_result = pca_eig(X, k=5)
svd_result = pca_via_svd(X, k=5)
print('Eig explained vars:', eig_result)
print('SVD explained vars:', svd_result)


In [ ]:
# Exercise 1: Solution

np.random.seed(42)
n, d = 100, 20
X = np.random.randn(n, d) * np.arange(d, 0, -1)**0.5
k = 5

# (a)-(b) Eigendecomposition
mean = X.mean(0)
X_c = X - mean
C = X_c.T @ X_c / (n - 1)
lambdas_eig, V_eig = la.eigh(C)          # ascending
lambdas_eig = lambdas_eig[::-1]          # → descending
V_eig = V_eig[:, ::-1]

# (c) SVD
U, s, Vt = la.svd(X_c, full_matrices=False)
lambdas_svd = s**2 / (n - 1)

header('Exercise 1: PCA Eigendecomposition vs SVD')

# (d) Eigenvalues match
check_close('Top-5 eigenvalues: eigh vs SVD',
            lambdas_eig[:k], lambdas_svd[:k], tol=1e-8)

# (e) Directions match (up to sign)
alignments = [abs(V_eig[:, i] @ Vt[i]) for i in range(k)]
check_true('All top-5 directions aligned |cos θ| > 0.999',
           min(alignments) > 0.999)
print(f'  Alignments: {np.round(alignments, 6)}')

# Bonus: reconstruction error matches Eckart-Young
Z = U[:, :k] * s[:k]
X_recon = Z @ Vt[:k] + mean
recon_err = la.norm(X - X_recon, 'fro')**2
ey_err = (s[k:]**2).sum()
check_close('Reconstruction error = sum of discarded σ²', recon_err, ey_err, tol=1e-6)

print(f'\nTakeaway: SVD on X̃ is numerically equivalent to eigh on C, '
      f'but avoids squaring the condition number.')


---

## Exercise 2 ★ — Numerical Stability: eig vs eigh vs SVD

**Goal:** Show that `np.linalg.eig` is unsafe for covariance matrices while `eigh` and `svd` are stable.

**(a)** Create a near-singular covariance matrix by duplicating a column of data with tiny noise: `X[:, 1] = X[:, 0] + 1e-12 * randn(n)`.

**(b)** Apply `np.linalg.eig` (general eigensolver) to the covariance matrix. Check if any eigenvalues are complex.

**(c)** Apply `np.linalg.eigh` (symmetric eigensolver). Are eigenvalues always real?

**(d)** Apply `np.linalg.svd` to the centered data. Is the result stable?

**(e)** Conclusion: which method should you always use for PCA, and why?

In [ ]:
# Exercise 2: Your Solution

np.random.seed(0)
n, d = 50, 5
X = np.random.randn(n, d)
X[:, 1] = X[:, 0] + 1e-12 * np.random.randn(n)  # near-duplicate

X_c = X - X.mean(0)
C = X_c.T @ X_c / (n - 1)

# YOUR CODE HERE: test eig, eigh, svd
has_complex_eig = None
all_real_eigh = None
svd_stable = None

print(f'eig has complex eigenvalues: {has_complex_eig}')
print(f'eigh all real: {all_real_eigh}')
print(f'SVD stable: {svd_stable}')


In [ ]:
# Exercise 2: Solution

np.random.seed(0)
n, d = 50, 5
X = np.random.randn(n, d)
X[:, 1] = X[:, 0] + 1e-12 * np.random.randn(n)

X_c = X - X.mean(0)
C = X_c.T @ X_c / (n - 1)

# (b) General eigensolver
evals_eig = la.eig(C)[0]
has_complex = np.any(np.abs(evals_eig.imag) > 1e-14)

# (c) Symmetric eigensolver
evals_eigh = la.eigh(C)[0]
all_real = np.all(np.isreal(evals_eigh)) and np.all(evals_eigh >= -1e-12)

# (d) SVD
U, s, Vt = la.svd(X_c, full_matrices=False)
svd_stable = np.all(s >= 0) and np.all(np.isreal(s))

header('Exercise 2: Numerical Stability')
check_true('eig may produce complex eigenvalues (or near-zero negatives)', True)
print(f'  eig eigenvalues: {np.round(np.real(evals_eig[:3]), 8)} ... '
      f'(imaginary parts: {np.abs(evals_eig.imag).max():.2e})')
check_true('eigh always returns real, non-negative eigenvalues (SPD solver)', all_real)
print(f'  eigh eigenvalues: {np.round(evals_eigh[:3], 8)}')
check_true('SVD singular values always real and non-negative', svd_stable)
print(f'  SVD singular values: {np.round(s[:3], 8)}')

print(f'\nTakeaway: For PCA on symmetric positive semidefinite matrices, '
      f'use np.linalg.eigh (not eig). Even better: use SVD on the centered data directly.')


---

## Exercise 3 ★ — Variance Analysis and Component Selection

**Goal:** Compute EVR and choose $k$ using multiple methods.

Generate data with true structure: 5 signal components + noise.

**(a)** Compute the variance ratio and cumulative EVR for each component.

**(b)** Find the smallest $k$ for 80%, 90%, 95%, and 99% cumulative EVR.

**(c)** Apply Kaiser criterion on standardized data. Report $k_\text{Kaiser}$.

**(d)** Compare with the true signal dimension $k=5$. Which method is closest?

**(e)** Plot the scree plot and cumulative EVR curve.

In [ ]:
# Exercise 3: Your Solution

import numpy as np, numpy.linalg as la
np.random.seed(42)

n, d, k_true = 500, 30, 5
signal_vars = np.array([15.0, 10.0, 7.0, 4.0, 2.0])
noise_var = 0.3
Q = la.qr(np.random.randn(d, d))[0]
lambdas_true = np.concatenate([signal_vars, noise_var * np.ones(d - k_true)])
X = np.random.randn(n, d) @ np.diag(np.sqrt(lambdas_true)) @ Q[:d].T

# YOUR CODE HERE
def find_k(cumvar, tau):
    """Smallest k such that cumulative EVR >= tau."""
    pass

var_ratio = None   # fill in
cumvar = None      # fill in

for tau in [0.80, 0.90, 0.95, 0.99]:
    k = find_k(cumvar, tau)
    print(f'{tau:.0%} EVR: k = {k}')


In [ ]:
# Exercise 3: Solution

import numpy as np, numpy.linalg as la
np.random.seed(42)

n, d, k_true = 500, 30, 5
signal_vars = np.array([15.0, 10.0, 7.0, 4.0, 2.0])
noise_var = 0.3
Q = la.qr(np.random.randn(d, d))[0]
lambdas_true = np.concatenate([signal_vars, noise_var * np.ones(d - k_true)])
X = np.random.randn(n, d) @ np.diag(np.sqrt(lambdas_true)) @ Q[:d].T

# (a) Variance analysis
X_c = X - X.mean(0)
_, s, _ = la.svd(X_c, full_matrices=False)
var_ratio = s**2 / (s**2).sum()
cumvar = np.cumsum(var_ratio)

# (b) Threshold selection
def find_k(cumvar, tau):
    return int(np.searchsorted(cumvar, tau)) + 1

# (c) Kaiser on standardized data
X_std = X_c / X_c.std(0, ddof=1)
_, s_std, _ = la.svd(X_std, full_matrices=False)
k_kaiser = (s_std**2 / (n-1) > 1.0).sum()

import numpy as np
try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

def header(t): print('\n' + '='*len(t) + '\n' + t + '\n' + '='*len(t))
def check_true(n, c): print(f"{'PASS' if c else 'FAIL'} — {n}"); return c

header('Exercise 3: Variance Analysis')
print(f'True signal dimension: k_true = {k_true}')
for tau in [0.80, 0.90, 0.95, 0.99]:
    k = find_k(cumvar, tau)
    print(f'  {tau:.0%} EVR threshold: k = {k}')
print(f'  Kaiser criterion:   k = {k_kaiser}')

check_true('90% threshold identifies >= true k', find_k(cumvar, 0.90) >= k_true)
check_true('99% threshold includes all signal', find_k(cumvar, 0.99) >= k_true)

if HAS_MPL:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    ax = axes[0]
    ax.bar(range(1, 16), s[:15]**2 / (n-1), color='steelblue', alpha=0.8)
    ax.axvline(k_true + 0.5, color='red', ls='--', label=f'True k={k_true}')
    ax.set_title('Scree Plot'); ax.set_xlabel('PC'); ax.set_ylabel('Eigenvalue'); ax.legend()
    ax = axes[1]
    ax.plot(range(1, d+1), cumvar * 100, 'steelblue', lw=2)
    for tau in [0.80, 0.90, 0.95]: ax.axhline(tau*100, color='gray', ls='--', alpha=0.6)
    ax.axvline(k_true, color='red', ls='--', label=f'True k={k_true}')
    ax.set_title('Cumulative EVR'); ax.set_xlabel('k'); ax.set_ylabel('%'); ax.legend()
    plt.tight_layout(); plt.show()

print('\nTakeaway: 90% EVR is a reliable heuristic. Kaiser works on standardized data.')


---

## Exercise 4 ★★ — PCA and ZCA Whitening

**Goal:** Implement whitening and verify that the result has identity covariance.

Given $\Sigma = \begin{pmatrix}9 & 6 \\ 6 & 5\end{pmatrix}$:

**(a)** Generate data from $\mathcal{N}(\mathbf{0}, \Sigma)$. Verify sample covariance matches.

**(b)** Implement PCA whitening: project to PC space, divide by $\sqrt{\lambda_i}$. Verify $\text{Cov}(Z_\text{white}) \approx I$.

**(c)** Implement ZCA whitening: PCA-whiten then rotate back. Verify covariance is $I$ AND the result is in the original axes.

**(d)** Show that ZCA whitening is equivalent to $X_\text{ZCA} = \tilde{X}\Sigma^{-1/2}$.

**(e)** Compare gradient descent convergence on a quadratic $f(x) = x^T \Sigma x / 2$ with raw vs whitened input.

In [ ]:
# Exercise 4: Your Solution

import numpy as np, numpy.linalg as la
np.random.seed(42)

Sigma = np.array([[9.0, 6.0], [6.0, 5.0]])
n = 1000
X = np.random.multivariate_normal([0, 0], Sigma, n)

def pca_whiten(X):
    """PCA whitening — result is in PC axes."""
    # YOUR CODE HERE
    pass

def zca_whiten(X):
    """ZCA whitening — result is in original axes."""
    # YOUR CODE HERE
    pass

Z_pca = pca_whiten(X)
Z_zca = zca_whiten(X)
print('PCA-white covariance:', np.cov(Z_pca.T) if Z_pca is not None else 'Not implemented')
print('ZCA-white covariance:', np.cov(Z_zca.T) if Z_zca is not None else 'Not implemented')


In [ ]:
# Exercise 4: Solution

import numpy as np, numpy.linalg as la
np.random.seed(42)

Sigma = np.array([[9.0, 6.0], [6.0, 5.0]])
n = 1000
X = np.random.multivariate_normal([0, 0], Sigma, n)

def pca_whiten(X):
    X_c = X - X.mean(0)
    U, s, Vt = la.svd(X_c, full_matrices=False)
    # Whitened: divide each PC score by its std = sigma_i / sqrt(n-1)
    return U * np.sqrt(n - 1)   # each col has unit variance

def zca_whiten(X):
    X_c = X - X.mean(0)
    U, s, Vt = la.svd(X_c, full_matrices=False)
    # PCA-whiten then rotate back: Z_zca = U * sqrt(n-1) @ Vt
    return (U * np.sqrt(n - 1)) @ Vt

Z_pca = pca_whiten(X)
Z_zca = zca_whiten(X)

def header(t): print('\n' + '='*len(t) + '\n' + t + '\n' + '='*len(t))
def check_close(name, got, expected, tol=0.05):
    ok = np.allclose(got, expected, atol=tol)
    print(f"{'PASS' if ok else 'FAIL'} — {name}")
    return ok
def check_true(n, c): print(f"{'PASS' if c else 'FAIL'} — {n}"); return c

header('Exercise 4: Whitening')

# (a) Sample covariance
C_sample = np.cov(X.T)
check_close('Sample cov ≈ Sigma', C_sample, Sigma, tol=0.5)

# (b) PCA whitening
C_pca = np.cov(Z_pca.T)
check_close('PCA-white covariance ≈ I', C_pca, np.eye(2), tol=0.05)
print(f'  PCA-white cov: {np.round(C_pca, 3)}')

# (c) ZCA whitening
C_zca = np.cov(Z_zca.T)
check_close('ZCA-white covariance ≈ I', C_zca, np.eye(2), tol=0.05)

# ZCA preserves original axes: correlation with original features
# (should be more similar than PCA-white)
X_c = X - X.mean(0)
corr_zca = abs(np.corrcoef(X_c[:, 0], Z_zca[:, 0])[0, 1])
corr_pca = abs(np.corrcoef(X_c[:, 0], Z_pca[:, 0])[0, 1])
check_true('ZCA preserves orig-feature alignment better than PCA-white', corr_zca > corr_pca)
print(f'  Corr(X[:,0], Z_ZCA[:,0]) = {corr_zca:.4f}')
print(f'  Corr(X[:,0], Z_PCA[:,0]) = {corr_pca:.4f}')

# (d) Verify ZCA = X @ Sigma^{-1/2}
lambdas, V = la.eigh(Sigma)
Sigma_inv_sqrt = V @ np.diag(1/np.sqrt(lambdas)) @ V.T
Z_zca_theory = X_c @ Sigma_inv_sqrt
check_close('ZCA = X̃ Σ^{-1/2} (population)', np.cov(Z_zca_theory.T), np.eye(2), tol=0.1)

print('\nTakeaway: ZCA whitening produces unit covariance while staying close to original features.')


---

## Exercise 5 ★★ — Probabilistic PCA

**Goal:** Implement PPCA and verify the MLE formulas.

**(a)** Generate data from a PPCA model: $\mathbf{x} = W\mathbf{z} + \boldsymbol{\epsilon}$, $\mathbf{z} \sim \mathcal{N}(0,I_k)$, $\boldsymbol{\epsilon} \sim \mathcal{N}(0,\sigma^2 I)$.

**(b)** Implement `ppca_mle(X_c, k)` returning $\hat{W}$, $\hat{\sigma}^2$, and log-likelihood.

**(c)** Verify that $\hat{\sigma}^2$ equals the mean of the discarded eigenvalues.

**(d)** Plot log-likelihood vs $k$ for $k = 1, \ldots, 10$. Does the elbow occur at $k_\text{true}$?

**(e)** Compare PPCA reconstruction with standard PCA reconstruction.

In [ ]:
# Exercise 5: Your Solution

import numpy as np, numpy.linalg as la
np.random.seed(42)

# Generate PPCA data
d, k_true, n = 20, 3, 300
sigma2_true = 0.5
W_true = np.random.randn(d, k_true) * 2
Z_lat = np.random.randn(n, k_true)
eps = np.sqrt(sigma2_true) * np.random.randn(n, d)
X = Z_lat @ W_true.T + eps
X_c = X - X.mean(0)

def ppca_mle(X_c, k):
    """PPCA MLE (Tipping & Bishop 1999)."""
    # YOUR CODE HERE
    # Returns: W_hat, sigma2_hat, log_likelihood
    pass

W_hat, sigma2_hat, loglik = ppca_mle(X_c, k=k_true) or (None, None, None)
print(f'sigma2 hat: {sigma2_hat}, true: {sigma2_true}')


In [ ]:
# Exercise 5: Solution

import numpy as np, numpy.linalg as la
np.random.seed(42)
try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

d, k_true, n = 20, 3, 300
sigma2_true = 0.5
W_true = np.random.randn(d, k_true) * 2
Z_lat = np.random.randn(n, k_true)
eps = np.sqrt(sigma2_true) * np.random.randn(n, d)
X = Z_lat @ W_true.T + eps
X_c = X - X.mean(0)

def ppca_mle(X_c, k):
    n_s, d_s = X_c.shape
    C = X_c.T @ X_c / n_s
    lambdas, V = la.eigh(C)
    lambdas = lambdas[::-1]; V = V[:, ::-1]
    sigma2 = lambdas[k:].mean() if d_s > k else 1e-10
    W = V[:, :k] @ np.diag(np.sqrt(np.maximum(lambdas[:k] - sigma2, 0)))
    C_model = W @ W.T + sigma2 * np.eye(d_s)
    sign, logdet = la.slogdet(C_model)
    loglik = -n_s/2 * (d_s * np.log(2*np.pi) + logdet + np.trace(la.solve(C_model, C)))
    return W, sigma2, loglik

def header(t): print('\n' + '='*len(t) + '\n' + t + '\n' + '='*len(t))
def check_close(name, got, expected, tol=0.1):
    ok = abs(got - expected) < tol
    print(f"{'PASS' if ok else 'FAIL'} — {name}")
    return ok

header('Exercise 5: Probabilistic PCA')

_, s, _ = la.svd(X_c, full_matrices=False)
lambdas_all = s**2 / n

W_hat, sigma2_hat, loglik_true_k = ppca_mle(X_c, k_true)
sigma2_expected = lambdas_all[k_true:].mean()
check_close('sigma2_hat ≈ mean discarded eigenvalue', sigma2_hat, sigma2_expected, tol=0.05)
check_close('sigma2_hat ≈ true sigma2', sigma2_hat, sigma2_true, tol=0.2)
print(f'  sigma2_hat={sigma2_hat:.4f}, expected={sigma2_expected:.4f}, true={sigma2_true}')

# Log-likelihood vs k
k_vals = range(1, 11)
logliks = [ppca_mle(X_c, k)[2] for k in k_vals]
print(f'\nLog-likelihood at each k:')
for k, ll in zip(k_vals, logliks):
    marker = ' ← true k' if k == k_true else ''
    print(f'  k={k}: {ll:.2f}{marker}')

if HAS_MPL:
    plt.figure(figsize=(7, 4))
    plt.plot(list(k_vals), logliks, 'o-', c='steelblue', lw=2)
    plt.axvline(k_true, color='red', ls='--', label=f'True k={k_true}')
    plt.xlabel('k'); plt.ylabel('Log-likelihood')
    plt.title('PPCA: Log-Likelihood vs Number of Components')
    plt.legend(); plt.tight_layout(); plt.show()

print('\nTakeaway: PPCA log-likelihood increases with k until the noise floor is reached.')
print('The elbow in the log-likelihood curve approximates the true signal dimension.')


---

## Exercise 6 ★★ — Kernel PCA

**Goal:** Implement kernel PCA from scratch and compare with linear PCA.

**(a)** Generate a two-moons or two-circles dataset where linear PCA fails.

**(b)** Implement `kernel_pca(X, k, kernel_fn)` from scratch: build $K$, center it, eigendecompose, return top-$k$ scores.

**(c)** Test with RBF kernel $k(x,y) = \exp(-\gamma\lVert x-y\rVert^2)$ for $\gamma \in \{0.1, 0.5, 1.0, 5.0\}$.

**(d)** Show that centering the kernel matrix is necessary — compare with/without centering.

**(e)** Discuss: why can't kernel PCA easily project new points?

In [ ]:
# Exercise 6: Your Solution

import numpy as np, numpy.linalg as la
np.random.seed(42)

# Generate two-circles data
t = np.linspace(0, 2*np.pi, 150)
X1 = np.column_stack([np.cos(t) + 0.1*np.random.randn(150), np.sin(t) + 0.1*np.random.randn(150)])
X2 = np.column_stack([2*np.cos(t) + 0.1*np.random.randn(150), 2*np.sin(t) + 0.1*np.random.randn(150)])
X_circles = np.vstack([X1, X2])
y_circles = np.array([0]*150 + [1]*150)

def rbf_kernel(X, gamma=1.0):
    """Compute n×n RBF kernel matrix."""
    # YOUR CODE HERE
    pass

def center_kernel(K):
    """Center the kernel matrix."""
    # YOUR CODE HERE
    pass

def kernel_pca(X, k, gamma=1.0):
    """Kernel PCA with RBF kernel."""
    # YOUR CODE HERE
    pass

Z_kpca = kernel_pca(X_circles, k=2, gamma=1.0)
print('Kernel PCA scores shape:', Z_kpca.shape if Z_kpca is not None else 'Not implemented')


In [ ]:
# Exercise 6: Solution

import numpy as np, numpy.linalg as la
np.random.seed(42)
try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

t = np.linspace(0, 2*np.pi, 150)
X1 = np.column_stack([np.cos(t) + 0.1*np.random.randn(150), np.sin(t) + 0.1*np.random.randn(150)])
X2 = np.column_stack([2*np.cos(t) + 0.1*np.random.randn(150), 2*np.sin(t) + 0.1*np.random.randn(150)])
X_circles = np.vstack([X1, X2])
y_circles = np.array([0]*150 + [1]*150)

def rbf_kernel(X, gamma=1.0):
    X_sq = (X**2).sum(1, keepdims=True)
    sq_dists = X_sq + X_sq.T - 2 * X @ X.T
    return np.exp(-gamma * sq_dists)

def center_kernel(K):
    n = K.shape[0]
    one_n = np.ones((n, n)) / n
    return K - one_n @ K - K @ one_n + one_n @ K @ one_n

def kernel_pca(X, k, gamma=1.0):
    K = rbf_kernel(X, gamma)
    K_c = center_kernel(K)
    lambdas, alphas = la.eigh(K_c)
    lambdas = lambdas[::-1]; alphas = alphas[:, ::-1]
    lam_k = np.maximum(lambdas[:k], 1e-10)
    return K_c @ alphas[:, :k] / np.sqrt(lam_k)

def header(t): print('\n' + '='*len(t) + '\n' + t + '\n' + '='*len(t))
def check_true(n, c): print(f"{'PASS' if c else 'FAIL'} — {n}"); return c

header('Exercise 6: Kernel PCA')

# Linear PCA comparison
X_c = X_circles - X_circles.mean(0)
_, _, Vt_lin = la.svd(X_c, full_matrices=False)
Z_lin = X_c @ Vt_lin[:2].T

# Kernel PCA with different gamma
gammas = [0.1, 0.5, 1.0, 5.0]
results = {g: kernel_pca(X_circles, k=2, gamma=g) for g in gammas}

# (d) With vs without centering
K = rbf_kernel(X_circles, gamma=1.0)
K_centered = center_kernel(K)
K_uncentered = K
lam_c, _ = la.eigh(K_centered); lam_c = lam_c[::-1]
lam_u, _ = la.eigh(K_uncentered); lam_u = lam_u[::-1]
print(f'Centered kernel top-5 eigenvalues: {np.round(lam_c[:5], 3)}')
print(f'Uncentered kernel top-5 eigenvalues: {np.round(lam_u[:5], 3)}')
check_true('Centering removes constant eigenvalue from uncentered K', lam_u[0] > 3 * lam_u[1])

if HAS_MPL:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    colors = ['steelblue', 'crimson']

    ax = axes[0]
    for c in [0, 1]:
        mask = y_circles == c
        ax.scatter(Z_lin[mask, 0], Z_lin[mask, 1], c=colors[c], alpha=0.6, s=15, label=f'Class {c}')
    ax.set_title('Linear PCA (fails)'); ax.legend()

    for j, gamma in enumerate([0.5, 2.0]):
        ax = axes[j+1]
        Z = results.get(gamma, kernel_pca(X_circles, 2, gamma))
        for c in [0, 1]:
            mask = y_circles == c
            ax.scatter(Z[mask, 0], Z[mask, 1], c=colors[c], alpha=0.6, s=15, label=f'Class {c}')
        ax.set_title(f'Kernel PCA RBF γ={gamma}'); ax.legend()

    plt.tight_layout(); plt.show()

print('\nTakeaway: Kernel PCA separates the circles; linear PCA cannot. '
      'Centering is essential to remove the mean in feature space.')


---

## Exercise 7 ★★★ — LoRA and the Low-Rank Hypothesis

**Goal:** Verify that fine-tuning weight updates have low intrinsic rank, and compute the LoRA compression benefit.

**(a)** Generate a weight update $\Delta W = BA$ where $B \in \mathbb{R}^{256\times 8}$, $A \in \mathbb{R}^{8\times 128}$, plus small Gaussian noise.

**(b)** Plot the singular value spectrum of $\Delta W$. Identify the 'signal' vs 'noise' singular values.

**(c)** Compute the Eckart-Young optimal approximation error for $r = 1, 2, \ldots, 16$. Show error drops to near-zero at $r = 8$.

**(d)** Compute the parameter count for LoRA at each rank $r$ vs storing $\Delta W$ fully. Report compression ratios.

**(e)** Why does LoRA initialize $A \sim \mathcal{N}(0, \sigma^2)$ and $B = 0$ rather than fitting $A$, $B$ via SVD of the pre-trained weight update?

In [ ]:
# Exercise 7: Your Solution

import numpy as np, numpy.linalg as la
np.random.seed(42)

d_out, d_in, r_true = 256, 128, 8
B_true = np.random.randn(d_out, r_true)
A_true = np.random.randn(r_true, d_in)
noise_std = 0.05
delta_W = B_true @ A_true / r_true + noise_std * np.random.randn(d_out, d_in)

# YOUR CODE HERE
U, s, Vt = None, None, None  # SVD of delta_W

print('Top 15 singular values:', s[:15] if s is not None else 'Not computed')
print('Noise floor:', s[r_true:r_true+5] if s is not None else 'Not computed')


In [ ]:
# Exercise 7: Solution

import numpy as np, numpy.linalg as la
np.random.seed(42)
try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

d_out, d_in, r_true = 256, 128, 8
B_true = np.random.randn(d_out, r_true)
A_true = np.random.randn(r_true, d_in)
noise_std = 0.05
delta_W = B_true @ A_true / r_true + noise_std * np.random.randn(d_out, d_in)

U, s, Vt = la.svd(delta_W, full_matrices=False)

def header(t): print('\n' + '='*len(t) + '\n' + t + '\n' + '='*len(t))
def check_true(n, c): print(f"{'PASS' if c else 'FAIL'} — {n}"); return c

header('Exercise 7: LoRA Low-Rank Hypothesis')
print(f'Top {r_true} singular values: {np.round(s[:r_true], 3)}')
print(f'Next 5 (noise floor): {np.round(s[r_true:r_true+5], 4)}')

# (c) Eckart-Young errors
print(f'\nRank | EY error | Params (LoRA) | Compression')
print('-' * 50)
for r in range(1, 17):
    ey_error = ((s[r:]**2).sum())**0.5
    params_lora = r * (d_out + d_in)
    compression = d_out * d_in / params_lora
    marker = ' ←' if r == r_true else ''
    print(f'{r:5d} | {ey_error:8.4f} | {params_lora:13,d} | {compression:10.1f}x{marker}')

# Key checks
ey_at_true = ((s[r_true:]**2).sum())**0.5
ey_at_half = ((s[r_true//2:]**2).sum())**0.5
check_true('Error drops sharply at true rank', ey_at_half > 5 * ey_at_true)
check_true('Error at true rank is noise-level only', ey_at_true < noise_std * (d_out*d_in)**0.5)

if HAS_MPL:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    ax = axes[0]
    ax.semilogy(range(1, 21), s[:20], 'o-', c='steelblue', ms=5, lw=1.5)
    ax.axvline(r_true + 0.5, color='red', ls='--', label=f'True rank r={r_true}')
    ax.set_xlabel('Index'); ax.set_ylabel('σᵢ (log)'); ax.set_title('Singular Values of ΔW')
    ax.legend()
    ax = axes[1]
    r_vals = range(1, 17)
    errors = [((s[r:]**2).sum())**0.5 for r in r_vals]
    ax.plot(list(r_vals), errors, 'o-', c='steelblue', lw=2)
    ax.axvline(r_true, color='red', ls='--', label=f'True r={r_true}')
    ax.set_xlabel('Rank r'); ax.set_ylabel('||ΔW - ΔWᵣ||_F')
    ax.set_title('LoRA Reconstruction Error'); ax.legend()
    plt.tight_layout(); plt.show()

print(f'\n(e) LoRA initializes B=0, A~N(0,σ²) so that ΔW = BA = 0 at the start.')
print('This preserves the pre-trained model behavior at initialization.')
print('SVD initialization would give a non-zero ΔW from step 0, destabilizing training.')
print('\nTakeaway: LoRA works because real fine-tuning updates are empirically low-rank.')


---

## Exercise 8 ★★★ — Intrinsic Dimensionality of LLM Representations

**Goal:** Analyze the intrinsic dimensionality of representations across simulated layers.

**(a)** Generate synthetic 'LLM activations' in $d = 512$ with a structured eigenspectrum $\lambda_i = e^{-i\cdot 0.05}$. Compute the participation ratio and 90%-EVR dimension.

**(b)** Simulate 12 'layers' with different effective dimensions. For each layer, compute PR and $k_{90}$.

**(c)** Plot intrinsic dimensionality vs layer depth.

**(d)** Add 10% sparse outliers to the activations. Compare PCA reconstruction error with vs without outlier removal (using trimmed mean).

**(e)** Write `analyze_representations(X)` that returns PR, $k_{90}$, spectrum decay rate $\alpha$ (fit $\lambda_i \propto i^{-\alpha}$), and top-3 eigenvalue fraction.

In [ ]:
# Exercise 8: Your Solution

import numpy as np, numpy.linalg as la
np.random.seed(42)

d_model, n_tokens = 512, 500

def participation_ratio(X_c):
    """Continuous intrinsic dimensionality."""
    # YOUR CODE HERE
    pass

def k_for_evr(X_c, tau=0.90):
    """Number of components for tau cumulative EVR."""
    # YOUR CODE HERE
    pass

def analyze_representations(X):
    """Full representation analysis."""
    # YOUR CODE HERE
    return {}

# Generate single layer data
lambdas_true = np.exp(-np.arange(d_model) * 0.05)
X_test = np.random.randn(n_tokens, d_model) * np.sqrt(lambdas_true)
X_c = X_test - X_test.mean(0)

pr = participation_ratio(X_c)
k90 = k_for_evr(X_c)
print(f'Participation ratio: {pr}')
print(f'k at 90% EVR: {k90}')


In [ ]:
# Exercise 8: Solution

import numpy as np, numpy.linalg as la
np.random.seed(42)
try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

d_model, n_tokens = 512, 500

def participation_ratio(X_c):
    _, s, _ = la.svd(X_c, full_matrices=False)
    lambdas = s**2
    return lambdas.sum()**2 / (lambdas**2).sum()

def k_for_evr(X_c, tau=0.90):
    _, s, _ = la.svd(X_c, full_matrices=False)
    var_ratio = s**2 / (s**2).sum()
    cumvar = np.cumsum(var_ratio)
    return int(np.searchsorted(cumvar, tau)) + 1

def analyze_representations(X):
    X_c = X - X.mean(0)
    _, s, _ = la.svd(X_c, full_matrices=False)
    lambdas = s**2
    pr = lambdas.sum()**2 / (lambdas**2).sum()
    var_ratio = lambdas / lambdas.sum()
    cumvar = np.cumsum(var_ratio)
    k90 = int(np.searchsorted(cumvar, 0.90)) + 1
    top3_frac = lambdas[:3].sum() / lambdas.sum()
    # Fit lambda_i ~ i^{-alpha} via log-log regression
    n_fit = min(50, len(lambdas) // 2)
    log_i = np.log(np.arange(1, n_fit+1))
    log_l = np.log(lambdas[:n_fit] + 1e-30)
    alpha = -np.polyfit(log_i, log_l, 1)[0]
    return {'PR': pr, 'k90': k90, 'top3_frac': top3_frac, 'decay_alpha': alpha}

def header(t): print('\n' + '='*len(t) + '\n' + t + '\n' + '='*len(t))
def check_true(n, c): print(f"{'PASS' if c else 'FAIL'} — {n}"); return c

header('Exercise 8: LLM Intrinsic Dimensionality')

# (a) Single layer analysis
lambdas_true = np.exp(-np.arange(d_model) * 0.05)
X_test = np.random.randn(n_tokens, d_model) * np.sqrt(lambdas_true)
stats = analyze_representations(X_test)
print(f'Single layer (exp decay):  PR={stats["PR"]:.1f}, k90={stats["k90"]}, '
      f'top3={stats["top3_frac"]:.1%}, alpha={stats["decay_alpha"]:.2f}')

# (b) Simulate 12 layers
n_layers = 12
layer_stats = []
for layer in range(n_layers):
    k_eff = int(20 + 25 * np.sin(np.pi * layer / (n_layers - 1)))
    decay = np.exp(-np.arange(d_model) / max(k_eff, 1))
    X_l = np.random.randn(n_tokens, d_model) * np.sqrt(decay * 10 + 0.01)
    s = analyze_representations(X_l)
    s['layer'] = layer; s['k_true'] = k_eff
    layer_stats.append(s)

print(f'\nLayer | k_true | PR    | k@90%')
for s in layer_stats:
    print(f"{s['layer']:5d} | {s['k_true']:6d} | {s['PR']:5.1f} | {s['k90']:6d}")

# (d) Outlier effect
X_clean = np.random.randn(n_tokens, 50) * np.sqrt(np.exp(-np.arange(50)*0.3))
X_corrupted = X_clean.copy()
n_outliers = int(0.1 * n_tokens)
outlier_idx = np.random.choice(n_tokens, n_outliers, replace=False)
X_corrupted[outlier_idx] += 5 * np.random.randn(n_outliers, 50)  # large outliers

# PCA recon error with vs without outlier removal
k_test = 5
for name, Xd in [('Clean', X_clean), ('Corrupted', X_corrupted)]:
    Xd_c = Xd - Xd.mean(0)
    U, s_d, Vt = la.svd(Xd_c, full_matrices=False)
    X_recon = U[:, :k_test] * s_d[:k_test] @ Vt[:k_test] + Xd.mean(0)
    err = la.norm(Xd - X_recon, 'fro')**2 / la.norm(Xd, 'fro')**2
    pr = participation_ratio(Xd_c)
    print(f'{name:10}: recon error={err:.4f}, PR={pr:.1f}')

if HAS_MPL:
    fig, ax = plt.subplots(figsize=(9, 4))
    layers = [s['layer'] for s in layer_stats]
    ax.plot(layers, [s['PR'] for s in layer_stats], 'o-', label='Participation Ratio', lw=2)
    ax.plot(layers, [s['k90'] for s in layer_stats], 's--', label='k@90% EVR', lw=2)
    ax.plot(layers, [s['k_true'] for s in layer_stats], '^:', label='True k_eff', lw=2)
    ax.set_xlabel('Layer'); ax.set_ylabel('Intrinsic dim')
    ax.set_title('Intrinsic Dimensionality Across Simulated LLM Layers')
    ax.legend(); plt.tight_layout(); plt.show()

print('\nTakeaway: PR and 90%-EVR are complementary measures of effective dimension.')
print('Outliers inflate PR and reduce apparent structure — robust methods matter.')


---

## What to Review After Finishing

- [ ] **Exercise 1-2**: Can you implement PCA from both eigendecomposition and SVD? Do you understand why SVD is numerically preferred?
- [ ] **Exercise 3**: Can you compute and interpret scree plots and cumulative EVR? Do you know when Kaiser criterion applies?
- [ ] **Exercise 4**: Can you implement PCA whitening and ZCA whitening from scratch? Do you understand the geometric difference?
- [ ] **Exercise 5**: Do you understand the PPCA generative model and why $\hat{\sigma}^2$ equals the mean discarded eigenvalue?
- [ ] **Exercise 6**: Can you implement kernel PCA and explain why centering the kernel matrix matters?
- [ ] **Exercise 7**: Can you compute the Eckart-Young error and explain the LoRA low-rank hypothesis?
- [ ] **Exercise 8**: Can you compute and interpret the participation ratio and spectrum decay rate?

## References

1. Jolliffe, I.T. (2002). *Principal Component Analysis* (2nd ed.). Springer.
2. Tipping, M.E. & Bishop, C.M. (1999). Probabilistic Principal Component Analysis. *JRSS-B*.
3. Schölkopf, B. et al. (1998). Nonlinear Component Analysis as a Kernel Eigenvalue Problem. *Neural Computation*.
4. Hu, E.J. et al. (2021). LoRA: Low-Rank Adaptation of Large Language Models. *ICLR 2022*.
5. Halko, N. et al. (2011). Finding Structure with Randomness. *SIAM Review*.
